In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [5]:
train_dataset = (torchvision.datasets.CIFAR10(root='/data', train=True, download=True, transform=transform))
test_dataset = (torchvision.datasets.CIFAR10(root='/data', train=False, download=True, transform=transform))

In [6]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=64)

CNN Build

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),     # Kernal size =2, stride =2
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.Fucntion_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)    # Flattening
        x = self.Fucntion_layers(x)

        return x

In [8]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimzer = optim.Adam(model.parameters())

In [16]:
# Training Model

train_losses = []
epochs = 10

for epoch in range(epochs):
    running_loss = 0.0
    for xb, yb, in train_loader:
        optimzer.zero_grad()

        output = model(xb) # Forward Propagation
        loss   = criterion(output, yb)  # Loss fnx
        loss.backward() # BackPropagation
        optimzer.step() # Update Params

        running_loss += loss.item()

epoch_train_loss = running_loss / len(train_loader)
train_losses.append(epoch_train_loss)

In [18]:
# Validation

val_losses = []
model.eval()
running_val_loss = 0.0

with torch.no_grad():   # no gradients compute
    for xb, yb in test_loader:
        output = model(xb)
        loss = criterion(output, yb)
        running_val_loss += loss

epoch_val_loss = running_val_loss / len(test_loader)
val_losses.append(epoch_val_loss)



In [19]:
# Evaluate our CNN

correct_labels = 0
total_labels   = 0

model.eval()

with torch.no_grad():
    for xb, yb in test_loader:
        output = model.forward(xb)
        _, predicted = torch.max(output, 1)

        correct_labels += (predicted == yb).sum().item()
        total_labels += yb.size(0)

print(f"Accuracy = {correct_labels / total_labels * 100}")

Accuracy = 74.69


In [26]:
# Print Train, Val Losses

train_losses, val_losses

([0.07416047277051094], [tensor(1.6631)])

In [27]:
# Save Model

torch.save(model.state_dict(), "best_cnn_model.pt")

In [31]:
# Load Model

model.load_state_dict(torch.load("best_cnn_model.pt"))

<All keys matched successfully>